In [ ]:


!pip install naijalingo

In [ ]:
from naijalingo import NaijaLingo

# Point to your local server
from naijalingo import NaijaLingo
API_KEY = "NAIJALINGO_API_KEY "
client = NaijaLingo(api_key=API_KEY)


In [2]:
# Test all endpoints
status = client.tts.health()
print("Health:", status.status, f"({status.speakers_loaded} speakers)")



Health: healthy (240 speakers)


In [ ]:
models = client.list_models()
print("Models:", [m.id for m in models])

In [ ]:

speakers = client.tts.list_speakers(language="yo")
print(f"Yoruba speakers: {len(speakers)}")

In [ ]:

speakers = client.tts.list_speakers(language="ig")
print(f"Igbo speakers: {len(speakers)}")

In [ ]:
speakers = client.tts.list_speakers(language="ha")
print(f"Hausa speakers: {len(speakers)}")

In [ ]:
speakers = client.tts.list_speakers(language="pcm")
print(f"Pidgin speakers: {len(speakers)}")

In [4]:
info = client.api_info()
print("API:", info.name, info.version)

API: 9jaLingo TTS API 1.0.0


In [5]:
# Generate speech
audio = client.tts.generate("How you dey?", voice="pcm", response_format="wav")
audio.save("files/pidgin_output.wav")
print(f"Generated {len(audio)} bytes")

Generated 169402 bytes


In [ ]:
# Generate speech
audio = client.tts.generate("How you dey?", voice="pcm", response_format="flac")
audio.save("files/pidgin_output.flac")
print(f"Generated {len(audio)} bytes")

In [ ]:
# With a specific speaker voice
audio = client.tts.generate(
    "Nnoo, kedu ka i mere?",
    voice="ig",
    speaker="adaeze_ig",
)
audio.save("files/adaeze_greeting.mp3")

In [ ]:
# # Stream to a file
# with open("long_speech.wav", "wb") as f:
#     for chunk in client.tts.stream("Very long text here...", voice="pcm"):
#         f.write(chunk)

# Or collect the full stream
long_text="""
        Life na one kind journey wey nobody fit fully understand. From the day person open eye for this world, the journey don start. 
        Some people go say life na race, some go say na school, others go talk say na battle. But the truth be say life na mixture of many things together. E get sweet time, e get bitter time, e get time wey everything go dey move smooth like fresh engine, and e get time wey everywhere go just scatter like market wey rain beat.
        """
stream = client.tts.stream(long_text, voice="pcm", speaker="ada_pcm")
audio = stream.collect()
audio.save("files/long_speech.wav")

In [ ]:
!pip install openai

In [ ]:
from pathlib import Path
from openai import OpenAI

client_openai = OpenAI(api_key="your-key")  # use any string for now
# speech_file_path = Path(__file__).parent / "speech.mp3"

with client_openai.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input=long_text,
    instructions="Speak in a cheerful and positive tone.",
) as response:
    response.stream_to_file("files/speech.mp3")

In [ ]:
import requests

with open("audio.wav", "rb") as f:
    resp = requests.post(
        "http://localhost:8000/v1/audio/clone",
        files={"audio": ("audio.wav", f, "audio/wav")},
        data={"text": "I believe say life dey treat you fine now, make you no forget your boy oo.", "voice": "pcm"},
           
    )

with open("cloned.wav", "wb") as f:
    f.write(resp.content)

In [6]:
from pathlib import Path

cwd = Path.cwd()

# If you're inside src/, go up one level
if (cwd / "files").exists():
    BASE_DIR = cwd
else:
    BASE_DIR = cwd.parent

audio_path = BASE_DIR / "files" / "pastor.aac"

audio = client.tts.clone(
    " Love your neighbor as yourself, so that your days will be long, and your Father in Heaven will bless you.",
    audio_file=str(audio_path),
    voice="pcm",
)

audio.save("cloned.wav")

# From file-like object
with open(audio_path, "rb") as f:
    audio = client.tts.clone("Hello!", audio_file=f, voice="pcm")

In [7]:
# List all speakers
speakers = client.tts.list_speakers()
for s in speakers:
    print(f"{s.id} — {s.language} ({s.gender})")

# Filter by language
yoruba_speakers = client.tts.list_speakers(language="yo")

# Get a specific speaker
speaker = client.tts.get_speaker("ada_pcm")
print(speaker.name, speaker.language)

Ada pcm


In [ ]:
# Get root API information
info = client.api_info()
print(info.name)        # "9jaLingo TTS API"
print(info.version)     # "1.0.0"
print(info.endpoints)   # dict of all available endpoints

# Get v1 service metadata
service = client.service_info()
print(service.name)             # "9jaLingo API v1"
print(service.speech_url)       # "/v1/audio/speech"
print(service.models_url)       # "/v1/models"